# Bearing Anomaly Detection — PCA Autoencoder (FFT Spectrum)

## 개요

슈레더 베어링 진동 센서의 FFT 스펙트럼 데이터를 PCA Autoencoder로 분석하여 베어링 이상을 탐지하는 **프로덕션 레벨** 모델입니다.

| 항목 | 내용 |
|------|------|
| **입력** | VIB-A/B 센서 6채널 (3축 × 2센서), FFT 1024 bins |
| **데이터** | 5,000 FFT 윈도우 (정상 ~90%, 이상 ~10%) |
| **모델** | PCA Autoencoder (LSTM Autoencoder 개념의 경량 구현) |
| **학습** | 정상 데이터만으로 비지도 학습 |
| **탐지** | 재구성 오차(MSE) 기반 이상 판정 |

### PCA Autoencoder 원리

```
정상 FFT 스펙트럼 → PCA Encoder (6144→50 차원 압축) → Latent → Inverse PCA Decoder (50→6144 복원)
                                                                    ↓
              정상이면 복원 잘 됨 → 오차 작음 → 정상 판정
              이상이면 복원 못 함 → 오차 큼  → 이상 판정!
```

### 베어링 결함 주파수

| 결함 유형 | 약어 | A축 주파수 | B축 주파수 |
|:---:|:---:|:---:|:---:|
| 외륜 결함 | BPFO | 71.2 Hz | 47.5 Hz |
| 내륜 결함 | BPFI | 108.8 Hz | 72.5 Hz |
| 볼 결함 | BSF | 46.4 Hz | 30.9 Hz |

---
## Step 0. Library Install & Import

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_fscore_support
)

print('Libraries loaded successfully.')

---
## Step 1. FFT Spectrum Data Generation (n_samples=5000)

VIB-A/B 센서의 3축(x, y, z) 진동 데이터를 FFT 변환한 주파수 도메인 스펙트럼을 생성합니다.

**데이터 구성:**
- **정상 (~90%)**: 1X, 2X, 3X 회전 고조파 + 배경 노이즈만 존재
- **이상 (~10%)**: 베어링 결함 주파수(BPFO, BPFI, BSF) 피크가 추가로 출현

**주요 파라미터:**
- 샘플링 주파수: 10 kHz → 나이퀴스트: 5,000 Hz
- FFT bins: 1024 (0 ~ 5000 Hz)
- A축 RPM: 1200 → 1X = 20 Hz
- B축 RPM: 800 → 1X = 13.3 Hz

In [ ]:
def generate_fft_data(n_samples=5000, n_bins=1024, seed=42):
    """
    FFT 스펙트럼 데이터 생성 — 베어링 이상 탐지용.
    VIB-A/B 센서의 3축 진동을 FFT 변환한 결과를 시뮬레이션.

    Returns:
        dict:
            'VIB_A_fft': (n_samples, 1024, 3) — A축 x,y,z FFT 파워 스펙트럼
            'VIB_B_fft': (n_samples, 1024, 3) — B축 x,y,z FFT 파워 스펙트럼
            'labels': (n_samples,) — 0=정상, 1=베어링이상
            'freqs': (1024,) — 주파수 축 (Hz), 0~5000Hz
    """
    rng = np.random.default_rng(seed)

    # 샘플링 주파수 및 주파수 축
    fs = 10000  # 10 kHz
    freqs = np.linspace(0, fs / 2, n_bins)  # 0 ~ 5000 Hz

    # 슈레더 축별 RPM 및 회전 주파수
    _BASE_RPM_A = 1200.0
    _BASE_RPM_B = 800.0
    f1x_a = _BASE_RPM_A / 60.0  # 20 Hz
    f1x_b = _BASE_RPM_B / 60.0  # 13.33 Hz

    # 베어링 결함 특성 주파수 (볼 베어링 근사)
    bpfo_a = f1x_a * 3.56   # 외륜 결함 (71.2 Hz)
    bpfi_a = f1x_a * 5.44   # 내륜 결함 (108.8 Hz)
    bsf_a  = f1x_a * 2.32   # 볼 결함 (46.4 Hz)

    bpfo_b = f1x_b * 3.56   # 47.5 Hz
    bpfi_b = f1x_b * 5.44   # 72.5 Hz
    bsf_b  = f1x_b * 2.32   # 30.9 Hz

    # 이상 비율 (~10%)
    n_anomaly = int(n_samples * 0.10)
    labels = np.zeros(n_samples, dtype=int)
    anomaly_indices = rng.choice(n_samples, size=n_anomaly, replace=False)
    labels[anomaly_indices] = 1

    def _freq_peak(freqs, center_freq, bandwidth=2.0, amplitude=1.0):
        """특정 주파수에 가우시안 피크 생성"""
        return amplitude * np.exp(-0.5 * ((freqs - center_freq) / bandwidth) ** 2)

    def _make_spectrum(is_anomaly, base_freq, bpfo, bpfi, bsf, axis_scale):
        """단일 축 FFT 파워 스펙트럼 생성"""
        spec = np.zeros(n_bins)

        # 1X 회전 주파수 피크
        spec += _freq_peak(freqs, base_freq, 1.5, rng.uniform(8, 12) * axis_scale)
        # 2X, 3X 고조파
        spec += _freq_peak(freqs, base_freq * 2, 1.5, rng.uniform(3, 5) * axis_scale)
        spec += _freq_peak(freqs, base_freq * 3, 2.0, rng.uniform(1, 3) * axis_scale)

        # 배경 노이즈 (1/f 스펙트럼)
        bg = 0.5 * axis_scale / (1 + freqs / 100)
        spec += bg + rng.exponential(0.1 * axis_scale, n_bins)

        if is_anomaly:
            # 베어링 결함 주파수 피크 추가
            severity = rng.uniform(0.5, 1.0)
            spec += _freq_peak(freqs, bpfo, 2.0, rng.uniform(5, 15) * severity * axis_scale)
            spec += _freq_peak(freqs, bpfi, 2.0, rng.uniform(3, 10) * severity * axis_scale)
            spec += _freq_peak(freqs, bsf, 1.5, rng.uniform(2, 8) * severity * axis_scale)
            # 결함 고조파 (2X, 3X)
            for h in [2, 3]:
                spec += _freq_peak(freqs, bpfo * h, 3.0,
                                   rng.uniform(1, 5) * severity * axis_scale)
            # 전체 노이즈 플로어 상승
            spec += rng.uniform(0.5, 2.0) * severity * axis_scale

        return np.maximum(spec, 0)

    # 6채널 FFT 생성 (A: x,y,z / B: x,y,z)
    vib_a_fft = np.zeros((n_samples, n_bins, 3))
    vib_b_fft = np.zeros((n_samples, n_bins, 3))
    axis_scales = [1.0, 0.8, 0.5]  # x > y > z (축별 감도 차이)

    for i in range(n_samples):
        is_anom = labels[i] == 1
        for ax_idx, scale in enumerate(axis_scales):
            vib_a_fft[i, :, ax_idx] = _make_spectrum(
                is_anom, f1x_a, bpfo_a, bpfi_a, bsf_a, scale)
            vib_b_fft[i, :, ax_idx] = _make_spectrum(
                is_anom, f1x_b, bpfo_b, bpfi_b, bsf_b, scale * 0.85)

    return {
        'VIB_A_fft': vib_a_fft,
        'VIB_B_fft': vib_b_fft,
        'labels': labels,
        'freqs': freqs,
    }

print('generate_fft_data() 정의 완료.')

In [ ]:
# FFT 데이터 생성 (5000 윈도우, 프로덕션 레벨)
fft_data = generate_fft_data(n_samples=5000, n_bins=1024, seed=42)

vib_a_fft = fft_data['VIB_A_fft']  # (5000, 1024, 3)
vib_b_fft = fft_data['VIB_B_fft']  # (5000, 1024, 3)
labels    = fft_data['labels']       # (5000,) 0=정상, 1=이상
freqs     = fft_data['freqs']        # (1024,) Hz

n_samples = len(labels)
n_bins = vib_a_fft.shape[1]

print(f'생성 완료: {n_samples}개 FFT 윈도우')
print(f'VIB_A shape: {vib_a_fft.shape}  (samples, bins, axes)')
print(f'VIB_B shape: {vib_b_fft.shape}')
print(f'주파수 범위: {freqs[0]:.1f} ~ {freqs[-1]:.1f} Hz')
print(f'정상: {(labels==0).sum()}개 ({(labels==0).mean()*100:.1f}%)')
print(f'이상: {(labels==1).sum()}개 ({(labels==1).mean()*100:.1f}%)')

---
## Step 2. Data Visualization

FFT 스펙트럼의 정상 vs 이상 패턴을 시각적으로 비교합니다.

### 2-1. Normal vs Anomaly FFT Spectrum (A-axis X channel)

정상 스펙트럼에는 1X, 2X, 3X 회전 고조파만 보이고, 이상 스펙트럼에는 **BPFO, BPFI, BSF** 결함 주파수 피크가 추가로 나타납니다.

In [ ]:
# 대표 정상/이상 샘플 선택
normal_indices = np.where(labels == 0)[0]
anomaly_indices = np.where(labels == 1)[0]
sample_normal = normal_indices[0]
sample_anomaly = anomaly_indices[0]

# 베어링 결함 주파수 정의 (A축)
f1x_a = 1200.0 / 60.0  # 20 Hz
bearing_freqs_a = {
    '1X (20Hz)': f1x_a,
    '2X (40Hz)': f1x_a * 2,
    '3X (60Hz)': f1x_a * 3,
    'BSF (46.4Hz)': f1x_a * 2.32,
    'BPFO (71.2Hz)': f1x_a * 3.56,
    'BPFI (108.8Hz)': f1x_a * 5.44,
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Normal spectrum
axes[0].plot(freqs, vib_a_fft[sample_normal, :, 0], 'b-', linewidth=0.8, alpha=0.8)
axes[0].set_title('Normal FFT Spectrum — VIB-A X-axis', fontsize=13, fontweight='bold')
axes[0].set_ylabel('FFT Power')
axes[0].grid(True, alpha=0.3)

# Anomaly spectrum
axes[1].plot(freqs, vib_a_fft[sample_anomaly, :, 0], 'r-', linewidth=0.8, alpha=0.8)
axes[1].set_title('Anomaly FFT Spectrum — VIB-A X-axis (bearing defect peaks visible)',
                   fontsize=13, fontweight='bold')
axes[1].set_ylabel('FFT Power')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].grid(True, alpha=0.3)

# 결함 주파수 마커 표시
for ax in axes:
    for name, fval in bearing_freqs_a.items():
        color = 'green' if 'X (' in name else 'red'
        ls = '--' if 'X (' in name else ':'
        ax.axvline(x=fval, color=color, linestyle=ls, alpha=0.6, linewidth=0.8)
        ax.text(fval + 1, ax.get_ylim()[1] * 0.85, name, fontsize=7,
                color=color, rotation=90, va='top')

for ax in axes:
    ax.set_xlim(0, 250)

plt.tight_layout()
plt.show()

### 2-2. Multi-channel Overview (6 channels)

VIB-A 3축(x, y, z) + VIB-B 3축(x, y, z) = **총 6채널**의 FFT 스펙트럼을 한눈에 비교합니다.
파란색 = 정상, 빨간색 = 이상.

In [ ]:
channel_names = ['VIB-A X', 'VIB-A Y', 'VIB-A Z', 'VIB-B X', 'VIB-B Y', 'VIB-B Z']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for idx, (ax, ch_name) in enumerate(zip(axes.flat, channel_names)):
    if idx < 3:
        normal_spec = vib_a_fft[sample_normal, :, idx]
        anomaly_spec = vib_a_fft[sample_anomaly, :, idx]
    else:
        normal_spec = vib_b_fft[sample_normal, :, idx - 3]
        anomaly_spec = vib_b_fft[sample_anomaly, :, idx - 3]

    ax.plot(freqs, normal_spec, 'b-', linewidth=0.6, alpha=0.7, label='Normal')
    ax.plot(freqs, anomaly_spec, 'r-', linewidth=0.6, alpha=0.7, label='Anomaly')
    ax.set_title(ch_name, fontsize=11, fontweight='bold')
    ax.set_xlim(0, 250)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[1, 1].set_xlabel('Frequency (Hz)', fontsize=11)
fig.suptitle('6-Channel FFT Spectrum Overview — Normal (blue) vs Anomaly (red)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3. Feature Engineering

6채널(VIB-A 3축 + VIB-B 3축)의 1024 bins FFT 스펙트럼을 **단일 특징 벡터**로 변환합니다.

```
VIB-A (1024 × 3) + VIB-B (1024 × 3) → 평탄화 → 6144차원 벡터
→ StandardScaler 정규화
```

In [ ]:
# 6채널 FFT를 단일 특징 행렬로 결합
# VIB-A (n, 1024, 3) → reshape → (n, 3072)
# VIB-B (n, 1024, 3) → reshape → (n, 3072)
# 결합 → (n, 6144)
X_a = vib_a_fft.reshape(n_samples, -1)  # (5000, 3072)
X_b = vib_b_fft.reshape(n_samples, -1)  # (5000, 3072)
X_all = np.hstack([X_a, X_b])            # (5000, 6144)

print(f'VIB-A 특징: {X_a.shape}')
print(f'VIB-B 특징: {X_b.shape}')
print(f'결합 특징:  {X_all.shape}')

# StandardScaler 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

print(f'\n정규화 후 평균: {X_scaled.mean():.6f}')
print(f'정규화 후 표준편차: {X_scaled.std():.6f}')

---
## Step 4. Train/Test Split

**핵심: 정상 데이터만으로 학습** (비지도 학습)

- 학습 데이터: 정상 데이터의 70%
- 테스트 데이터: 나머지 정상 30% + 전체 이상 데이터
- 테스트 시 정상/이상이 섞여 있어 실제 운영 환경을 모사

In [ ]:
# 정상/이상 데이터 분리
normal_mask = labels == 0
X_normal = X_scaled[normal_mask]
X_anomaly = X_scaled[~normal_mask]

# 정상 데이터의 70%를 학습, 30%를 테스트
train_size = int(len(X_normal) * 0.7)
X_train = X_normal[:train_size]
X_test_normal = X_normal[train_size:]

# 테스트: 나머지 정상 + 전체 이상
X_test = np.vstack([X_test_normal, X_anomaly])
test_labels = np.concatenate([
    np.zeros(len(X_test_normal)),
    np.ones(len(X_anomaly))
])

print(f'학습 데이터: {X_train.shape[0]}건 (정상만)')
print(f'테스트 데이터: {X_test.shape[0]}건')
print(f'  - 정상: {len(X_test_normal)}건')
print(f'  - 이상: {len(X_anomaly)}건')
print(f'  - 이상 비율: {len(X_anomaly)/len(X_test)*100:.1f}%')

---
## Step 5. PCA Autoencoder Model

PCA를 이용하여 LSTM Autoencoder의 핵심 개념을 구현합니다.

| 실제 LSTM Autoencoder | PCA 시뮬레이션 |
|:---:|:---:|
| Encoder (LSTM layers) | PCA transform (6144 → 50) |
| Latent space | 50개 주성분 |
| Decoder (LSTM layers) | PCA inverse_transform (50 → 6144) |
| Reconstruction error | MSE(원본 - 복원) |

**n_components=50**: 6144차원 입력을 50차원으로 압축합니다. 정상 패턴의 주요 변동만 보존되며, 이상 패턴의 미세한 결함 주파수는 손실됩니다.

In [ ]:
# PCA Autoencoder: 정상 데이터만으로 학습
N_COMPONENTS = 50

pca = PCA(n_components=N_COMPONENTS)
pca.fit(X_train)  # 정상 데이터만으로 학습!

explained_var = pca.explained_variance_ratio_.sum() * 100
print(f'PCA 학습 완료')
print(f'차원: {X_train.shape[1]} → {N_COMPONENTS} (압축률: {N_COMPONENTS/X_train.shape[1]*100:.2f}%)')
print(f'설명된 분산: {explained_var:.1f}%')

# Encoder → Decoder (정상 학습 데이터)
train_encoded = pca.transform(X_train)       # Encoder: 6144 → 50
train_decoded = pca.inverse_transform(train_encoded)  # Decoder: 50 → 6144

# 학습 데이터의 재구성 오차
train_mse = np.mean((X_train - train_decoded) ** 2, axis=1)

# 임계값: 학습 데이터 오차의 95th percentile
threshold = np.percentile(train_mse, 95)

print(f'\n학습 데이터 재구성 오차:')
print(f'  평균: {train_mse.mean():.6f}')
print(f'  중앙값: {np.median(train_mse):.6f}')
print(f'  95th pctl (threshold): {threshold:.6f}')
print(f'  최대: {train_mse.max():.6f}')

---
## Step 6. Anomaly Detection

테스트 데이터에 대해 재구성 오차를 계산하고, 임계값(threshold)을 기준으로 이상을 판정합니다.

- **재구성 오차 > threshold** → 이상 판정
- **이상 점수(Anomaly Score)**: 0~1로 정규화하여 등급 분류에 활용

In [ ]:
# 테스트 데이터 재구성
test_encoded = pca.transform(X_test)
test_decoded = pca.inverse_transform(test_encoded)

# 재구성 오차 (MSE)
test_mse = np.mean((X_test - test_decoded) ** 2, axis=1)

# 이상 판정
test_predicted = (test_mse > threshold).astype(int)

# 이상 점수 (0~1 정규화)
def compute_anomaly_scores(mse_values, threshold):
    """재구성 오차를 0~1 이상 점수로 변환"""
    scores = np.zeros_like(mse_values)
    # 정상 범위: 0 ~ threshold → 0 ~ 0.3
    normal_mask = mse_values <= threshold
    scores[normal_mask] = 0.3 * (mse_values[normal_mask] / (threshold + 1e-10))
    # 이상 범위: threshold ~ → 0.3 ~ 1.0
    anomaly_mask = mse_values > threshold
    if anomaly_mask.any():
        max_error = max(mse_values.max(), threshold * 5)
        scores[anomaly_mask] = 0.3 + 0.7 * (
            (mse_values[anomaly_mask] - threshold) /
            (max_error - threshold + 1e-10)
        )
    return np.clip(scores, 0, 1)

train_scores = compute_anomaly_scores(train_mse, threshold)
test_scores = compute_anomaly_scores(test_mse, threshold)

# 결과 요약
n_detected = test_predicted.sum()
print(f'임계값 (threshold): {threshold:.6f}')
print(f'\n테스트 이상 감지: {n_detected}건 / {len(X_test)}건 ({n_detected/len(X_test)*100:.1f}%)')
print(f'실제 이상 수: {(test_labels==1).sum()}건')
print(f'\n테스트 재구성 오차:')
print(f'  정상 평균: {test_mse[test_labels==0].mean():.6f}')
print(f'  이상 평균: {test_mse[test_labels==1].mean():.6f}')
print(f'  이상/정상 비율: {test_mse[test_labels==1].mean() / test_mse[test_labels==0].mean():.1f}x')

---
## Step 7. Result Visualization

### 7-1. Normal vs Anomaly Reconstructed Spectrum Comparison

정상 샘플은 원본과 복원이 거의 일치하지만, 이상 샘플은 **결함 주파수 피크 부분에서 복원 오차**가 크게 발생합니다.

In [ ]:
# 대표 정상/이상 샘플 (테스트 데이터 기준)
test_normal_scores = test_scores[:len(X_test_normal)]
test_anomaly_scores = test_scores[len(X_test_normal):]
best_normal_idx = np.argmin(test_normal_scores)
worst_anomaly_idx = len(X_test_normal) + np.argmax(test_anomaly_scores)

# 복원된 스펙트럼을 원래 스케일로 역변환
X_test_orig = scaler.inverse_transform(X_test)
X_test_recon = scaler.inverse_transform(test_decoded)

# A축 X채널만 추출 (첫 1024 bins)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Normal
axes[0].plot(freqs, X_test_orig[best_normal_idx, :1024], 'b-',
             linewidth=0.8, alpha=0.8, label='Original')
axes[0].plot(freqs, X_test_recon[best_normal_idx, :1024], 'g--',
             linewidth=0.8, alpha=0.8, label='Reconstructed')
axes[0].set_title(f'Normal Sample — Reconstruction (Score: {test_scores[best_normal_idx]:.4f})',
                   fontsize=13, fontweight='bold')
axes[0].set_ylabel('FFT Power')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 250)

# Anomaly
axes[1].plot(freqs, X_test_orig[worst_anomaly_idx, :1024], 'r-',
             linewidth=0.8, alpha=0.8, label='Original')
axes[1].plot(freqs, X_test_recon[worst_anomaly_idx, :1024], 'm--',
             linewidth=0.8, alpha=0.8, label='Reconstructed')
axes[1].set_title(f'Anomaly Sample — Reconstruction (Score: {test_scores[worst_anomaly_idx]:.4f})',
                   fontsize=13, fontweight='bold')
axes[1].set_ylabel('FFT Power')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 250)

# 베어링 결함 주파수 표시
for ax in axes:
    for name, fval in [('BPFO', f1x_a*3.56), ('BPFI', f1x_a*5.44), ('BSF', f1x_a*2.32)]:
        ax.axvline(x=fval, color='orange', linestyle=':', alpha=0.6, linewidth=0.8)
        ax.text(fval + 1, ax.get_ylim()[1] * 0.9, name, fontsize=8, color='orange')

plt.tight_layout()
plt.show()

### 7-2. Reconstruction Error Time Series + Threshold

전체 테스트 데이터의 재구성 오차를 시계열로 표시합니다. 빨간 점선(임계값)을 초과하는 샘플이 이상으로 감지됩니다.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

time_idx = np.arange(len(test_mse))
colors = np.where(test_labels == 1, 'red', 'blue')

ax.scatter(time_idx, test_mse, c=colors, s=6, alpha=0.5,
           label='Reconstruction Error (blue=Normal, red=True Anomaly)')
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2,
           label=f'Threshold ({threshold:.6f})')
ax.fill_between(time_idx, 0, test_mse,
                where=test_mse > threshold,
                color='red', alpha=0.15)

ax.set_title('Reconstruction Error — Higher = More Abnormal',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Reconstruction Error (MSE)')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-3. Anomaly Score Distribution

학습 정상 / 테스트 정상 / 테스트 이상 세 그룹의 재구성 오차 분포를 비교합니다. 이상 데이터는 높은 오차 영역에 분포합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(train_mse, bins=60, alpha=0.6, color='green',
        label='Train (Normal)', density=True)
ax.hist(test_mse[test_labels == 0], bins=60, alpha=0.5, color='blue',
        label='Test (Normal)', density=True)
ax.hist(test_mse[test_labels == 1], bins=40, alpha=0.5, color='red',
        label='Test (Anomaly)', density=True)
ax.axvline(x=threshold, color='red', linestyle='--', linewidth=2,
           label=f'Threshold ({threshold:.6f})')

ax.set_title('Reconstruction Error Distribution — Train Normal vs Test Normal vs Test Anomaly',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-4. ROC Curve & Confusion Matrix

모델의 탐지 성능을 ROC 곡선과 혼동 행렬로 평가합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(test_labels, test_mse)
roc_auc = auc(fpr, tpr)

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Confusion Matrix
cm = confusion_matrix(test_labels, test_predicted)
im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Normal', 'Anomaly'])
axes[1].set_yticklabels(['Normal', 'Anomaly'])

# 숫자 표시
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        axes[1].text(j, i, str(cm[i, j]),
                     ha='center', va='center', fontsize=16, color=color, fontweight='bold')

plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'AUC: {roc_auc:.4f}')

### 7-5. Anomaly Grade Bands (Good / Warning / Danger)

이상 점수를 4단계 등급으로 분류합니다:
- **정상 (Good)**: 0.0 ~ 0.3
- **주의 (Caution)**: 0.3 ~ 0.6
- **경고 (Warning)**: 0.6 ~ 0.8
- **위험 (Danger)**: 0.8 ~ 1.0

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 등급별 배경색
ax.axhspan(0.0, 0.3, alpha=0.08, color='green')
ax.axhspan(0.3, 0.6, alpha=0.08, color='gold')
ax.axhspan(0.6, 0.8, alpha=0.08, color='orange')
ax.axhspan(0.8, 1.0, alpha=0.08, color='red')

# 등급별 색상 매핑
grade_colors = np.where(test_scores < 0.3, 'green',
               np.where(test_scores < 0.6, 'gold',
               np.where(test_scores < 0.8, 'orange', 'red')))

# 정상/이상 라벨별 마커
ax.scatter(time_idx[test_labels == 0], test_scores[test_labels == 0],
           c=grade_colors[test_labels == 0], s=8, alpha=0.6, marker='o', label='Normal label')
ax.scatter(time_idx[test_labels == 1], test_scores[test_labels == 1],
           c=grade_colors[test_labels == 1], s=25, alpha=0.8, marker='^', label='Anomaly label')

# 등급 경계선
ax.axhline(y=0.3, color='green', linestyle='--', alpha=0.7, label='Good/Caution (0.3)')
ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.7, label='Caution/Warning (0.6)')
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Warning/Danger (0.8)')

# 등급 텍스트
ax.text(len(test_scores) * 0.98, 0.15, 'GOOD', ha='right', fontsize=11,
        color='green', fontweight='bold')
ax.text(len(test_scores) * 0.98, 0.45, 'CAUTION', ha='right', fontsize=11,
        color='goldenrod', fontweight='bold')
ax.text(len(test_scores) * 0.98, 0.7, 'WARNING', ha='right', fontsize=11,
        color='orange', fontweight='bold')
ax.text(len(test_scores) * 0.98, 0.9, 'DANGER', ha='right', fontsize=11,
        color='red', fontweight='bold')

ax.set_title('Anomaly Score with Grade Bands (triangle = true anomaly)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Anomaly Score (0~1)')
ax.set_ylim(-0.02, 1.02)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 등급별 통계
good_cnt = (test_scores < 0.3).sum()
caution_cnt = ((test_scores >= 0.3) & (test_scores < 0.6)).sum()
warning_cnt = ((test_scores >= 0.6) & (test_scores < 0.8)).sum()
danger_cnt = (test_scores >= 0.8).sum()

print(f'등급별 분류:')
print(f'  Good    (0.0~0.3): {good_cnt}건 ({good_cnt/len(test_scores)*100:.1f}%)')
print(f'  Caution (0.3~0.6): {caution_cnt}건 ({caution_cnt/len(test_scores)*100:.1f}%)')
print(f'  Warning (0.6~0.8): {warning_cnt}건 ({warning_cnt/len(test_scores)*100:.1f}%)')
print(f'  Danger  (0.8~1.0): {danger_cnt}건 ({danger_cnt/len(test_scores)*100:.1f}%)')

### 7-6. Per-Channel Error Contribution

이상 감지 시 **어떤 채널(센서 축)이 가장 크게 기여**했는지 분석합니다. 이를 통해 결함 위치(A축 vs B축, x/y/z 방향)를 추정할 수 있습니다.

In [ ]:
# 채널별 재구성 오차 계산 (6채널: A-x, A-y, A-z, B-x, B-y, B-z)
channel_names_full = ['VIB-A X', 'VIB-A Y', 'VIB-A Z', 'VIB-B X', 'VIB-B Y', 'VIB-B Z']
n_ch = 6
bins_per_ch = n_bins  # 1024

test_error_per_channel = np.zeros((len(X_test), n_ch))
for ch in range(n_ch):
    start = ch * bins_per_ch
    end = start + bins_per_ch
    test_error_per_channel[:, ch] = np.mean(
        (X_test[:, start:end] - test_decoded[:, start:end]) ** 2, axis=1
    )

# 정상 vs 이상 그룹 평균
normal_err = test_error_per_channel[test_labels == 0].mean(axis=0)
anomaly_err = test_error_per_channel[test_labels == 1].mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(n_ch)
width = 0.35
bars1 = ax.bar(x - width/2, normal_err, width, label='Normal', color='green', alpha=0.7)
bars2 = ax.bar(x + width/2, anomaly_err, width, label='Anomaly', color='red', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(channel_names_full, fontsize=10)
ax.set_title('Per-Channel Reconstruction Error: Normal vs Anomaly',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Reconstruction Error (MSE)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 채널별 기여도
print('채널별 이상 시 오차 기여도:')
total = anomaly_err.sum()
for name, err in zip(channel_names_full, anomaly_err):
    print(f'  {name:10s}: {err:.6f} ({err/total*100:.1f}%)')

---
## Step 8. Summary

### 탐지 성능 지표

In [ ]:
# Performance Metrics
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_predicted, average='binary'
)

tp = int(((test_labels == 1) & (test_predicted == 1)).sum())
fp = int(((test_labels == 0) & (test_predicted == 1)).sum())
fn = int(((test_labels == 1) & (test_predicted == 0)).sum())
tn = int(((test_labels == 0) & (test_predicted == 0)).sum())

print('=' * 60)
print('  Bearing Anomaly Detection — Performance Summary')
print('=' * 60)
print(f'''
  [탐지 성능]
    Precision: {precision:.4f}
    Recall:    {recall:.4f}
    F1-Score:  {f1:.4f}
    AUC:       {roc_auc:.4f}

  [혼동 행렬]
    True Positive:  {tp:5d}  |  False Positive: {fp:5d}
    False Negative: {fn:5d}  |  True Negative:  {tn:5d}

  [등급 분류]
    Good:    {good_cnt}건 ({good_cnt/len(test_scores)*100:.1f}%)
    Caution: {caution_cnt}건 ({caution_cnt/len(test_scores)*100:.1f}%)
    Warning: {warning_cnt}건 ({warning_cnt/len(test_scores)*100:.1f}%)
    Danger:  {danger_cnt}건 ({danger_cnt/len(test_scores)*100:.1f}%)
''')

### 모델 특성 요약

| 항목 | 내용 |
|------|------|
| **입력** | VIB-A/B 6채널 FFT 스펙트럼 (6144차원) |
| **모델** | PCA Autoencoder (6144 → 50 → 6144) |
| **학습 데이터** | 정상 운전 데이터만 사용 (비지도 학습) |
| **이상 탐지** | 재구성 오차 > 95th percentile threshold |
| **등급 분류** | Good / Caution / Warning / Danger (4단계) |
| **데이터 규모** | 5,000 FFT 윈도우 (프로덕션 레벨) |

### 실무 적용 참고사항

1. **데이터 수집**: 정상 운전 28일 이상의 FFT 스펙트럼 수집 → 학습
2. **실시간 적용**: 매 1초마다 FFT 계산 → 재구성 오차 산출 → 이상 점수 계산
3. **임계값 조정**: 현장 환경에 맞춰 95th percentile 대신 다른 값 사용 가능
4. **채널별 분석**: 어떤 센서 축에서 오차가 큰지 확인하여 결함 위치 추정
5. **결함 주파수**: BPFO/BPFI/BSF 피크 출현 여부로 결함 유형 식별 가능
6. **모델 업그레이드**: PCA → LSTM Autoencoder (TensorFlow/PyTorch)로 전환 시 시퀀스 패턴 학습 가능

In [ ]:
print('=' * 60)
print('  Bearing Anomaly Detection — Complete!')
print('=' * 60)
print(f'''
  [데이터]
    입력: VIB-A/B 6채널 FFT x {n_bins} bins = {n_ch * n_bins}차원
    학습: {X_train.shape[0]}건 (정상 운전 데이터만)
    테스트: {X_test.shape[0]}건 (정상 {len(X_test_normal)} + 이상 {len(X_anomaly)})

  [모델]
    구조: Encoder ({n_ch * n_bins} -> {N_COMPONENTS}) -> Latent ({N_COMPONENTS}) -> Decoder ({N_COMPONENTS} -> {n_ch * n_bins})
    학습 방식: 비지도 학습 (정상 데이터만으로 학습)
    설명 분산: {explained_var:.1f}%

  [이상 탐지 결과]
    임계값: {threshold:.6f}
    Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {roc_auc:.4f}

  [슈레더 적용 시]
    1. 정상 운전 28일간 FFT 스펙트럼 수집 -> 학습
    2. 실시간으로 매 1초마다 FFT 계산 -> 재구성 오차 산출
    3. 이상 점수가 임계값 초과 시 경보 발생
    4. 채널별 오차 분석으로 결함 위치 추정
    5. BPFO/BPFI/BSF 피크로 결함 유형 식별
''')